# Embeddings: PHATE or UMAP of a subsample

1. **Subsample**: up to 2,000 cells from each sample of `standard_adata.h5ad`, picked so rare cell states are kept, saved as `standard_adata_sub.h5ad` with every measurement. Running it again replaces that file, and with it any embeddings already made.
2. **Pick features and samples**: the picked features decide which cells are neighbours; then every feature gets a score for how smoothly it changes between neighbouring cells (lower is smoother). Leave DNA and EdU out if you want to colour the embedding by them afterwards, as a check.
3. **Run a grid of settings**: each run is one embedding (PHATE or UMAP, 2D or 3D, how many of the top-ranked picked features it uses, and the method's own settings). Every combination of the values you list is run, side by side. About 10 s per run on 8,000 cells.
4. **Look**: colour an embedding by any measurement, or by a column such as treatment or sample_ID.

✅ Final product: `standard_adata_sub.h5ad`, with each embedding in `.obsm` and the settings that made it in `.uns["embeddings"]`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

In [ ]:
import anndata as ad

from fouri.embedding import Method, embed_grid, make_runs, rank_features, subsample
from fouri.io import adata_sub_path
from fouri.plots import plot_embedding

### 1. Subsample

In [ ]:
sub = subsample(exp, cells_per_sample=2000, seed=42)

### 2. Pick features and samples

The table lists every feature by its score, best first; `picked` marks the ones you chose. An embedding uses the top-ranked picked features, as many as each run's `top_n`.

In [ ]:
# the features that decide which cells are neighbours (old 08's list, in experiment 1.3's panel)
PICKED = [
    "total_area_DNA",
    "pRB_RB_ratio",
    "R1_CyclinA2_nuc_mean",
    "R0_CDK2_nuc_mean",
    "R3_CyclinE1_nuc_mean",
    "R2_CyclinD1_nuc_mean",
    "R1_CDK4_nuc_mean",
    "R3_p21_nuc_mean",
    "R1_E2F1_nuc_mean",
    "R2_CyclinB1_nuc_mean",
    "R4_CDT1_nuc_mean",
    "R5_cMyc_nuc_mean",
]
# the samples whose cells are used (sample_ID in experiment.json); None uses every cell
SAMPLES = [1, 2, 3, 4, 5]
# how many neighbours each cell has when ranking the features
RANKING_K = 100

ranking = rank_features(sub, PICKED, samples=SAMPLES, k=RANKING_K)
ranking.to_frame().assign(picked=ranking.index.isin(PICKED)).head(30)

### 3. Run a grid of settings

List one or several values for each setting; every combination becomes one embedding.
- PHATE: `knn` (neighbours per cell), `t` (`'auto'` or a number), `gamma`.
- UMAP: `n_neighbors`, `min_dist`.

Running the same settings again replaces that embedding (the result is identical). The same settings with other picked features or samples are kept apart, with `_2`, `_3` added to the name.

In [ ]:
runs = make_runs(
    Method.PHATE,
    dims=[3],
    top_n=[10],  # how many of the top-ranked picked features each embedding uses
    settings={"knn": [80], "t": ["auto"], "gamma": [1]},  # old 08's settings
)
# UMAP as well, for example:
# runs += make_runs(
#     Method.UMAP, dims=[2], top_n=[10], settings={'n_neighbors': [15, 30], 'min_dist': [0.1]}
# )

keys = embed_grid(exp, runs, PICKED, samples=SAMPLES, ranking_k=RANKING_K)

### 4. Look at an embedding

Drag to turn a 3D plot, scroll to zoom, click a legend entry to hide or show a group. Cells of samples the embedding left out aren't drawn.

In [ ]:
sub = ad.read_h5ad(adata_sub_path(exp))
print("Embeddings in the file:", ", ".join(sub.obsm))

KEY = keys[0]
plot_embedding(sub, KEY, "R5_EdU_nuc_mean")

In [ ]:
plot_embedding(sub, KEY, "treatment")

In [ ]:
# the settings and features that made this embedding
dict(sub.uns["embeddings"][KEY])